In [2]:
iris = load_iris()

# Выбранные признаки:
# 2 - petal length (cm)
# 3 - petal width (cm)
FEATURE_IDX = [2, 3]

X = iris.data[:, FEATURE_IDX]
y = iris.target

feature_names = [iris.feature_names[i] for i in FEATURE_IDX]
class_names = list(iris.target_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (105, 2)
Test shape: (45, 2)


In [3]:
class GaussianBayesClassifier:
    """Байесовский классификатор с нормальными классовыми распределениями."""

    def __init__(self, reg=1e-6):
        self.reg = reg

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        self.priors_ = np.zeros(n_classes)
        self.means_ = np.zeros((n_classes, n_features))
        self.covs_ = np.zeros((n_classes, n_features, n_features))
        self.inv_covs_ = np.zeros_like(self.covs_)
        self.logdet_covs_ = np.zeros(n_classes)

        for k, c in enumerate(self.classes_):
            Xc = X[y == c]

            self.priors_[k] = Xc.shape[0] / n_samples
            self.means_[k] = Xc.mean(axis=0)

            cov = np.cov(Xc, rowvar=False, bias=False)
            if cov.ndim == 0:
                cov = np.array([[float(cov)]])

            # Регуляризация для устойчивого обращения
            cov = cov + self.reg * np.eye(n_features)

            self.covs_[k] = cov
            self.inv_covs_[k] = np.linalg.inv(cov)
            self.logdet_covs_[k] = np.linalg.slogdet(cov)[1]

        return self

    def class_log_likelihood(self, X):
        X = np.asarray(X, dtype=float)

        n_features = X.shape[1]
        n_classes = len(self.classes_)
        loglik = np.zeros((X.shape[0], n_classes))

        const = n_features * np.log(2.0 * np.pi)

        for k in range(n_classes):
            diff = X - self.means_[k]

            # Mahalanobis distance для каждого объекта
            mahal = np.einsum(
                "ij,jk,ik->i",
                diff,
                self.inv_covs_[k],
                diff
            )

            loglik[:, k] = -0.5 * (
                const + self.logdet_covs_[k] + mahal
            )

        return loglik

    def joint_log_likelihood(self, X):
        return np.log(self.priors_) + self.class_log_likelihood(X)

    def predict_log_proba(self, X):
        jll = self.joint_log_likelihood(X)

        # log-sum-exp для численной устойчивости
        max_jll = jll.max(axis=1, keepdims=True)
        log_sum_exp = max_jll + np.log(
            np.exp(jll - max_jll).sum(axis=1, keepdims=True)
        )

        return jll - log_sum_exp

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]


## Линии уровня плотности распределения классов

Для каждого класса построена плотность нормального распределения,
оценённая по обучающей выборке:

`p(x | y = c) = N(mu_c, Sigma_c)`


## Разделяющие границы байесовского классификатора

Граница между двумя классами `i` и `j` определяется равенством
лог-апостериорных оценок:

`log P(y = i | x) = log P(y = j | x)`

На графике ниже показаны области решений и линии уровня, соответствующие
разделяющим границам.


## Ошибки классификации

Объекты, на которых классификатор ошибся, выделены красными крестами.
